In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import pandas as pd
import re

from urllib.parse import quote
from pathlib import Path


# ============================================================
# PROJECT PATHS
# ============================================================

BASE_DIR = Path.cwd().parent
RAW_DIR = BASE_DIR / "data" / "raw"

RAW_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# CARREFOUR SCRAPER
# ============================================================

def scrape_carrefour_search(search_term):

    # --------------------------------------------------------
    # BUILD URL
    # --------------------------------------------------------

    base_url = (
        "https://www.carrefour.es/supermercado/"
        "la-despensa/lacteos/leche/cat20093/c"
    )

    encoded_term = quote(search_term)

    url = f"{base_url}?query={encoded_term}"

    print(f"\nSearching Carrefour for: {search_term}")
    print(f"URL: {url}")

    # --------------------------------------------------------
    # CHROME CONFIGURATION
    # --------------------------------------------------------

    options = Options()
    options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(options=options)

    try:

        # ----------------------------------------------------
        # OPEN PAGE
        # ----------------------------------------------------

        driver.get(url)

        wait = WebDriverWait(driver, 20)

        # ----------------------------------------------------
        # ACCEPT COOKIES
        # ----------------------------------------------------

        try:

            accept_cookies = wait.until(
                EC.element_to_be_clickable(
                    (
                        By.XPATH,
                        "//button[contains(., 'Aceptar')]"
                    )
                )
            )

            accept_cookies.click()

            print("Cookies accepted.")

        except:

            print(
                "Cookie button not found or already accepted."
            )

        # ----------------------------------------------------
        # WAIT FOR PRODUCT CARDS
        # ----------------------------------------------------

        wait.until(
            EC.presence_of_element_located(
                (
                    By.CSS_SELECTOR,
                    "li.product-card-list__item"
                )
            )
        )

        products = driver.find_elements(
            By.CSS_SELECTOR,
            "li.product-card-list__item"
        )

        print(
            f"Product cards found: {len(products)}"
        )

        # ----------------------------------------------------
        # EXTRACT PRODUCTS
        # ----------------------------------------------------

        products_data = []

        skipped = 0

        for product in products:

            # ------------------------------------------------
            # NAME
            # ------------------------------------------------

            name_elements = product.find_elements(
                By.CSS_SELECTOR,
                ".product-card__title-link"
            )

            if not name_elements:
                skipped += 1
                continue

            name_element = name_elements[0]

            name = name_element.text.strip()

            if not name:
                skipped += 1
                continue

            # ------------------------------------------------
            # URL
            # ------------------------------------------------

            product_url = name_element.get_attribute(
                "href"
            )

            # ------------------------------------------------
            # PRICE
            # ------------------------------------------------

            price_elements = product.find_elements(
                By.CSS_SELECTOR,
                ".product-card__price"
            )

            if not price_elements:
                print(
                    f"Skipping product without price: {name}"
                )

                skipped += 1
                continue

            price = (
                price_elements[0]
                .text
                .strip()
            )

            # ------------------------------------------------
            # BRAND
            # ------------------------------------------------

            parent_elements = product.find_elements(
                By.CSS_SELECTOR,
                ".product-card__parent"
            )

            if parent_elements:

                brand = parent_elements[0].get_attribute(
                    "brand"
                )

            else:

                brand = None

            # ------------------------------------------------
            # PRODUCT ID
            # ------------------------------------------------

            if product_url:

                match = re.search(
                    r"/(R-[^/]+)/p",
                    product_url
                )

                product_id = (
                    match.group(1)
                    if match
                    else None
                )

            else:

                product_id = None

            # ------------------------------------------------
            # PRICE PER UNIT
            # ------------------------------------------------

            unit_elements = product.find_elements(
                By.CSS_SELECTOR,
                ".product-card__price-per-unit"
            )

            if unit_elements:

                price_per_unit_raw = (
                    unit_elements[0]
                    .text
                    .strip()
                )

            else:

                price_per_unit_raw = None

            # ------------------------------------------------
            # STORE PRODUCT
            # ------------------------------------------------

            products_data.append(
                {
                    "searched_item": search_term,
                    "supermarket": "Carrefour",
                    "product_id": product_id,
                    "brand": brand,
                    "name": name,
                    "price": price,
                    "price_per_unit_raw": price_per_unit_raw,
                    "url": product_url
                }
            )

        print(
            f"Valid products extracted: "
            f"{len(products_data)}"
        )

        print(
            f"Cards skipped: {skipped}"
        )

        # ----------------------------------------------------
        # CREATE DATAFRAME
        # ----------------------------------------------------

        df = pd.DataFrame(products_data)

        if df.empty:
            print("No valid products extracted.")
            return df

        # ----------------------------------------------------
        # CLEAN PRICE
        #
        # "0,84 €" -> 0.84
        # ----------------------------------------------------

        df["price"] = (
            df["price"]
            .str.replace(
                "€",
                "",
                regex=False
            )
            .str.replace(
                ",",
                ".",
                regex=False
            )
            .str.strip()
            .astype(float)
        )

        # ----------------------------------------------------
        # UNIT
        #
        # "0,84 €/l" -> L
        # ----------------------------------------------------

        df["unit"] = (
            df["price_per_unit_raw"]
            .str.extract(
                r"€/([A-Za-zÁÉÍÓÚáéíóú\.]+)"
            )[0]
            .str.replace(
                ".",
                "",
                regex=False
            )
            .str.upper()
        )

        # ----------------------------------------------------
        # PRICE PER UNIT
        #
        # "0,84 €/l" -> 0.84
        # ----------------------------------------------------

        df["price_per_unit"] = (
            df["price_per_unit_raw"]
            .str.extract(
                r"([\d,]+)"
            )[0]
            .str.replace(
                ",",
                ".",
                regex=False
            )
        )

        df["price_per_unit"] = pd.to_numeric(
            df["price_per_unit"],
            errors="coerce"
        )

        # ----------------------------------------------------
        # REMOVE RAW COLUMN
        # ----------------------------------------------------

        df = df.drop(
            columns=[
                "price_per_unit_raw"
            ]
        )

        return df

    finally:

        driver.quit()


# ============================================================
# TEST
# ============================================================

df_carrefour = scrape_carrefour_search(
    "leche semidesnatada"
)


# ============================================================
# SHOW RESULTS
# ============================================================

print("\nCARREFOUR RESULTS:")

print(df_carrefour)


# ============================================================
# SAVE RESULTS
# ============================================================

if not df_carrefour.empty:

    output_file = (
        RAW_DIR /
        "carrefour_leche_semidesnatada.csv"
    )

    df_carrefour.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"\nDataset saved successfully in:\n"
        f"{output_file}"
    )


Searching Carrefour for: leche semidesnatada
URL: https://www.carrefour.es/supermercado/la-despensa/lacteos/leche/cat20093/c?query=leche%20semidesnatada
Cookies accepted.
Product cards found: 29
Skipping product without price: Leche semidesnatada Carrefour pack de 9 briks de 1 l.
Skipping product without price: Leche desnatada Pascual sin lactosa brik 1 l.
Valid products extracted: 6
Cards skipped: 23

CARREFOUR RESULTS:
         searched_item supermarket   product_id                      brand  \
0  leche semidesnatada   Carrefour  R-521007071                  CARREFOUR   
1  leche semidesnatada   Carrefour  R-521007079                        NaN   
2  leche semidesnatada   Carrefour  R-521006992                  CARREFOUR   
3  leche semidesnatada   Carrefour  R-714713105                  CARREFOUR   
4  leche semidesnatada   Carrefour  R-521007006                        NaN   
5  leche semidesnatada   Carrefour  R-521007075  CENTRAL LECHERA ASTURIANA   

                           